<div dir="rtl">

# الدرس 17: الانحدار المتعدد وتجهيز الميزات

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

## لماذا يتعثر النزول بالتدرّج بدون توحيد؟

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def path(a, lr, start, steps=25):
    p = np.array(start, dtype=float); pts = [p.copy()]
    for _ in range(steps):
        p = p - lr * np.array([2 * a * p[0], 2 * p[1]])
        pts.append(p.copy())
    return np.array(pts)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
W1, W2 = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-3, 3, 200))
for ax, a, lr, title in [(axes[0], 25, 0.038, "different scales"), (axes[1], 1, 0.3, "after scaling")]:
    ax.contour(W1, W2, a * W1**2 + W2**2, levels=12, cmap="Blues")
    pts = path(a, lr, [-2.5, 2.5])
    ax.plot(pts[:, 0], pts[:, 1], "o-", color="C1", ms=3, lw=1.5)
    ax.scatter([0], [0], marker="*", s=150, color="C2", zorder=3)
    ax.set_title(title); ax.set_xlabel("w1"); ax.set_ylabel("w2")
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

<div dir="rtl">

### الانحدار المتعدد بالمصفوفات

</div>

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n = 200
area = rng.uniform(50, 200, n)
rooms = rng.integers(1, 6, n)
age = rng.uniform(0, 40, n)
price = 20 * area + 50 * rooms - 15 * age + 300 + rng.normal(0, 100, n)

X = np.column_stack([area, rooms, age])
y = price
print("X shape:", X.shape)
print("column means:", X.mean(axis=0).round(1))

In [ ]:
def fit_gd(X, y, lr, steps):
    w, b = np.zeros(X.shape[1]), 0.0
    history = []
    for _ in range(steps):
        err = X @ w + b - y
        w -= lr * 2 * X.T @ err / len(y)     # كل الأوزان في سطر واحد
        b -= lr * 2 * err.mean()
        history.append(np.mean(err ** 2))
    return w, b, history

<div dir="rtl">

### بدون توحيد، ثم مع التوحيد

</div>

In [ ]:
w_raw, b_raw, h_raw = fit_gd(X, y, lr=5e-5, steps=1000)
print("raw   : MSE after 1000 steps =", round(h_raw[-1]), " b =", round(b_raw, 1))

w_bad, b_bad, h_bad = fit_gd(X, y, lr=6e-5, steps=50)
print("raw with lr=6e-5: MSE after 50 steps =", f"{h_bad[-1]:.1e}")

mu, sd = X.mean(axis=0), X.std(axis=0)
Xs = (X - mu) / sd
w_s, b_s, h_s = fit_gd(Xs, y, lr=0.1, steps=100)
print("scaled: MSE after 100 steps  =", round(h_s[-1]))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.4))
plt.plot(h_raw[:100], lw=2, label="raw features, lr = 5e-5")
plt.plot(h_s, lw=2, label="scaled features, lr = 0.1")
plt.yscale("log")
plt.xlabel("step"); plt.ylabel("MSE (log scale)")
plt.legend(); plt.grid(True)
plt.show()

<div dir="rtl">

### نرجع الأوزان إلى وحداتها الأصلية

</div>

In [ ]:
w_orig = w_s / sd
b_orig = b_s - np.sum(w_s * mu / sd)
print("learned  :", w_orig.round(2), round(b_orig, 1))
print("true rule:", [20, 50, -15], 300)

from sklearn.linear_model import LinearRegression
m = LinearRegression().fit(X, y)
print("sklearn  :", m.coef_.round(2), round(m.intercept_, 1))

<div dir="rtl">

### منحنى بنموذج خطي

</div>

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

x1 = np.linspace(-1, 6, 40)
y1 = 0.5 * x1**2 - 2 * x1 + 3 + rng.normal(0, 0.5, x1.size)
X1 = x1.reshape(-1, 1)

line = LinearRegression().fit(X1, y1)
curve = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression()).fit(X1, y1)

print("line  MSE:", round(np.mean((line.predict(X1) - y1) ** 2), 3))
print("curve MSE:", round(np.mean((curve.predict(X1) - y1) ** 2), 3))
print("curve weights (x, x^2):", curve[-1].coef_.round(3), " bias:", round(curve[-1].intercept_, 3))

plt.figure(figsize=(6, 3.4))
plt.scatter(x1, y1, s=20, color="C1", label="data")
plt.plot(x1, line.predict(X1), lw=2, label="linear")
plt.plot(x1, curve.predict(X1), lw=2.5, label="with x^2 feature")
plt.legend(); plt.grid(True)
plt.show()

<div dir="rtl">

### الترميز الأحادي

</div>

In [ ]:
import pandas as pd

df = pd.DataFrame({"district": ["Fatih", "Besiktas", "Kadikoy", "Fatih"],
                   "area": [90, 120, 100, 70]})
print(pd.get_dummies(df, columns=["district"], dtype=int))

from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder(handle_unknown="ignore")
print(enc.fit_transform(df[["district"]]).toarray())
print(enc.categories_)

<div dir="rtl">

## تمرين: تدرّج، ثم منحنى بدرجات مختلفة

الجزء الأول، على الورق:

$$X = \begin{bmatrix} 1 & 0 \\ 0 & 1 \\ 1 & 1 \end{bmatrix} \qquad y = \begin{bmatrix} 2 \\ 3 \\ 4 \end{bmatrix} \qquad w = \begin{bmatrix} 0 \\ 0 \end{bmatrix} \qquad b = 0$$

احسب التوقعات، والأخطاء، وتدرّج الأوزان، ومشتقة الإزاحة.

الجزء الثاني، بالكود:
استخدم نفس البيانات المنحنية من الدرس. درّب ثلاثة نماذج بميزات متعددة الحدود من الدرجة 1 و 2 و 10.
اطبع متوسط مربعات الأخطاء لكل نموذج. هل الدرجة 10 أفضل فعلاً؟

</div>

In [ ]:
for degree in [1, 2, 10]:
    # درّب نموذجاً هنا، واطبع الخطأ
    pass